# 🎓 수학교육평가론 — 협력학습 MVP (v4)

**Human 1명 + AI 학생 3명(민준·서연·연우)** 이 '소수와 합성수'를 함께 탐구하는 협력학습 세션.
학습자모델(v2.0) · 교수자모델을 구현하고, 발화 기반 CPS 태깅과 자기효능감 자기보고 설문까지 통합.

```
User ──▶ 학습자 분석(01) ──▶ Learner Model(v2.0) ──▶ Tutor Decision(02) ──▶ AI 학생 발화(03)
                                    │
                                    ├─ CPS 태깅(08) ─→ cps 카운터 +1
                                    └─ 자기효능감 설문(09) ─→ se_01~08 pre/post
```

## v4 핵심 변화
- **학습자모델 v2.0**: 인지 3개(task_achievement·math_communication·math_reasoning) + 정의 2개(cps·self_efficacy) + 확장 슬롯
- **3개 과제 구조**: Stage 1 개념 설명 → Stage 2 소수 찾기(30~50, 함정 33/39/49) → Stage 3 적응형 교과서 예제
- **PISA 2015 CPS 프레임워크**: shared_understanding · action_taking · team_organisation · repair_moves 4개 하위구인 누적 카운터 + 파생 레벨
- **Bandura(2006) 자기효능감**: 과제-특이적 8문항 Likert 4점, pre/post 직접 자기보고


## 1️⃣ 의존성 설치 + 저장소 준비

- `anthropic`, `gradio` 설치
- GitHub에서 코드 clone (최초 1회) 또는 `git pull` (재실행)
- **한글 폰트 NanumGothic** 은 `fonts/` 폴더에 동봉되어 별도 설치 불필요


In [ ]:
# 셀 1: 라이브러리 + 코드 저장소
\!pip install anthropic gradio -q

import os
REPO_URL = "https://github.com/hi-math/tobagi_v1.0.git"
REPO_DIR = "tobagi_v1_0"
if os.path.isdir(REPO_DIR):
    \!cd {REPO_DIR} && git pull -q
    print(f"🔄 기존 저장소 업데이트: {REPO_DIR}")
else:
    \!git clone -q {REPO_URL} {REPO_DIR}
    print(f"⬇️  저장소 클론 완료: {REPO_DIR}")


## 2️⃣ 경로 · 모델 설정

로컬 환경에서는 `BASE_PATH`를 저장소 절대경로로 바꾸세요 (예: `/content/drive/MyDrive/평가론 과제`).


In [ ]:
# 셀 2: 경로 및 모델 설정
BASE_PATH = "tobagi_v1_0"                    # 저장소 루트 (상대/절대 모두 가능)
MODEL     = "claude-sonnet-4-20250514"       # Claude 모델 식별자

import sys, os
_abs = os.path.abspath(BASE_PATH)
if _abs not in sys.path:
    sys.path.insert(0, _abs)

print(f"✅ BASE_PATH={BASE_PATH\!r}  (abs: {_abs})")
print(f"✅ MODEL={MODEL\!r}")


## 3️⃣ 부트스트랩 — 설정 로드 + 학습자모델 초기화 + Claude API 준비

`bootstrap()` 한 번으로 `CONFIG`(학습자·교수자·과제·도메인) + `PROMPTS`(9개 템플릿) + `learner_models`(user + ai_1~3) + `api`(Claude 래퍼) 네 가지를 받습니다.


In [ ]:
# 셀 3: 부트스트랩
from google.colab import userdata
from lib import bootstrap

apikey = ""

if len(apikey)>2:
  
    ctx = bootstrap(
        base_path=BASE_PATH,
        api_key=apikey,
        model=MODEL,
        setup_fonts=True,        # NanumGothic 폰트 matplotlib 등록
    )
else :
    ctx = bootstrap(
        base_path=BASE_PATH,
        api_key=userdata.get("CLAUDE_API_KEY"),
        model=MODEL,
        setup_fonts=True,        # NanumGothic 폰트 matplotlib 등록
)

CONFIG         = ctx["config"]
PROMPTS        = ctx["prompts"]
learner_models = ctx["learner_models"]
api            = ctx["api"]

print(f"✅ Task: {CONFIG['tasks']['task_title']}")
print(f"✅ Stages: {len(CONFIG['tasks']['stages'])}")
print(f"✅ AI 학생: {[p['name'] for p in CONFIG['personas']['ai_students'].values()]}")
print(f"✅ 학습자 모델 스키마 v{CONFIG['learner_model_schema'].get('schema_version','?')}")
print(f"   · 카테고리: {list(CONFIG['learner_model_schema'].get('categories', {}).keys())}")
print(f"   · 모델: {list(CONFIG['learner_model_schema']['models'].keys())}")
print(f"✅ 프롬프트: {list(PROMPTS.keys())}")
print(f"✅ 학습자 인스턴스: {list(learner_models.keys())}")


## 4️⃣ 학습자 모델 v2.0 구조 확인

초기화된 사용자 학습자 모델을 Markdown으로 렌더해 v2.0 스키마가 정상 반영되었는지 확인합니다.


In [ ]:
# 셀 4: 학습자 모델 스냅샷
from IPython.display import Markdown, display
from lib import user_model_markdown

# 사용자 모델 초기 상태
display(Markdown(user_model_markdown(CONFIG, learner_models)))

# AI 학생 페르소나도 한 눈에
for aid, info in CONFIG["personas"]["ai_students"].items():
    print(f"🤖 {aid} — {info['name']} ({info['level']}/{info['role']})")
    print(f"   · {info['description']}")


## 5️⃣ Gradio 대화형 UI 실행

- **왼쪽**: 채팅(나 ↔ 민준·서연·연우) + `▶ 다음 Stage` 버튼
- **오른쪽 탭**: 🕸️ 레이더 · 📈 변화 추이 · 🧠 학습자모델 · 🧭 교수자 Decision
- Colab에서는 `share=True`로 공용 링크(~72시간) 생성
- 세션 종료 후 아래 셀들에서 CPS 태깅·자기효능감 설문·추가 시각화가 가능합니다.


In [ ]:
# 셀 5: Gradio UI 기동
from lib import launch_ui

launch_ui(**ctx, share=True)


## 6️⃣ (데모) 사용자 발화 CPS 태깅

`prompts/08_cps_tagging.md`를 Claude에 실제로 호출해, 사용자 발화 하나를 PISA 2015 CPS 4개 하위구인으로 다중 태깅합니다.
결과는 `learner_models['user']['models']['cps']` 카운터에 가산됩니다.

> 한 발화가 여러 하위구인에 해당하면 모두 태그됩니다. `confidence < 0.6`은 자동 누락.


In [ ]:
# 셀 6: CPS 태깅 데모
from lib.llm_api import render_prompt, extract_json
from lib.learner_model import apply_cps_tags

# 대상 발화 (원하면 실제 대화 로그에서 골라쓰세요)
USER_UTTERANCE = "민준이 말대로 1은 약수가 1개라서 소수가 아니네. 그럼 2부터 차례대로 확인해 보자."

# 직전 대화 맥락 (세션에서 끌어오거나 수동 제공)
recent_dialogue = [
    "민준: 1은 약수 개수가 1개라서 소수가 아니야. 소수는 약수가 정확히 2개인 수거든.",
    "서연: 지금까지 얘기한 게 '소수는 약수가 2개인 수' 맞지?",
    "연우: 나는 진짜 모르겠는데… 그럼 2는?",
]

stage = 1
stage_info = CONFIG["tasks"]["stages"][stage - 1]
prompt = render_prompt(PROMPTS["cps_tagging"], {
    "task_title":       CONFIG["tasks"]["task_title"],
    "stage_title":      stage_info["title"],
    "user_utterance":   USER_UTTERANCE,
    "recent_dialogue":  "\n".join(recent_dialogue),
})

raw = api.call(prompt, max_tokens=800, temperature=0.2)
result = extract_json(raw)

print("📝 태깅 결과")
for t in result.get("tags", []):
    print(f"  · [{t['dimension']:<22}] conf={t['confidence']:.2f} — {t.get('evidence_feature','')}")
    print(f"      quote: {t.get('quote','')\!r}")
if result.get("none"):
    print("  (태그 없음)")
if result.get("note"):
    print(f"  note: {result['note']}")

# 학습자 모델에 반영
gained = apply_cps_tags(learner_models["user"], result, stage=stage, turn=0)
print(f"\n✅ cps 카운터에 {gained}건 가산")
for dk, dv in learner_models["user"]["models"]["cps"].items():
    print(f"  · {dk}: 누적 {dv['value']}회")


## 7️⃣ (데모) 수학 자기효능감 pre/post 자기보고

Bandura(2006) 과제-특이적 자기효능감 8문항. 발화 분석으로 대체하지 **않고**, 학습자가 직접 응답합니다.
아래는 `input()`으로 간단히 받는 버전(실제 UI는 Gradio/웹폼 권장).

- `phase="pre"`는 Stage 시작 직후
- `phase="post"`는 Stage 종료 시점
- 척도: 1(전혀 자신 없음) ~ 4(매우 자신 있음)


In [ ]:
# 셀 7: 자기효능감 설문 (pre + post 데모)
from lib.learner_model import apply_self_efficacy_responses

# se_01~se_08 문항 (09_self_efficacy_survey.md에 정의됨)
SE_ITEMS = [
    ("se_01", "어떤 수가 소수인지 합성수인지 판단할 수 있다.", 1),
    ("se_02", "어떤 수가 소수인지 합성수인지 이유와 함께 설명할 수 있다.", 1),
    ("se_03", "1이 왜 소수가 아닌지 설명할 수 있다.", 1),
    ("se_04", "어떤 수를 약수쌍으로 나타내어 소수인지 합성수인지 판단할 수 있다.", 2),
    ("se_05", "어떤 수를 배열(직사각형 만들기)로 생각해 소수인지 합성수인지 판단할 수 있다.", 2),
    ("se_06", "친구가 '홀수는 다 소수'라고 말했을 때, 반례를 들어 설명할 수 있다.", 2),
    ("se_07", "내 생각이 맞는지 확인하기 위해 다른 방법으로 다시 검토할 수 있다.", 3),
    ("se_08", "민준이나 서연이와 이야기하면서 내 생각을 수학적으로 분명하게 설명할 수 있다.", None),
]

# ★ 실제 사용자 응답 대신 샘플 응답 — 아래 두 dict를 바꿔 쓰시면 됩니다.
pre_responses  = {"se_01": 2, "se_02": 2, "se_03": 2, "se_04": 2, "se_05": 1, "se_06": 1, "se_07": 2, "se_08": 2}
post_responses = {"se_01": 3, "se_02": 3, "se_03": 4, "se_04": 3, "se_05": 2, "se_06": 2, "se_07": 3, "se_08": 3}

from datetime import datetime
now_iso = datetime.now().isoformat(timespec="seconds")

w_pre = apply_self_efficacy_responses(learner_models["user"], {
    "phase": "pre", "stage": 0, "timestamp": now_iso,
    "responses": pre_responses,
})
w_post = apply_self_efficacy_responses(learner_models["user"], {
    "phase": "post", "stage": 3, "timestamp": now_iso,
    "responses": post_responses,
})
print(f"✅ pre {w_pre}문항 · post {w_post}문항 기록됨")

# pre → post 변화 요약
se = learner_models["user"]["models"]["self_efficacy"]
for iid, item_text, st in SE_ITEMS:
    pre  = se[iid].get("pre")
    post = se[iid].get("post")
    delta = (post - pre) if (isinstance(pre,int) and isinstance(post,int)) else None
    dmark = "▲" if (delta or 0) > 0 else ("▼" if (delta or 0) < 0 else "■")
    print(f"  [{iid}] Stage{st}  pre={pre}  post={post}  {dmark} {delta:+d}" if delta is not None
          else f"  [{iid}] Stage{st}  pre={pre}  post={post}")


## 8️⃣ 학습자 모델 종합 시각화

세션 + CPS 태깅 + 자기효능감 응답이 반영된 최종 학습자 모델을 여러 방식으로 표시합니다.

- `user_model_markdown` — 카테고리별 Stage 등급·루브릭 힌트·CPS 파생 레벨·pre/post 평균
- `plot_radar_all` — 의사소통·추론 등 ordinal 차원 레이더
- `plot_user_history` — 회차별 변화 추이
- `cps_heatmap_figure` — Stage × CPS 4하위구인 누적 히트맵
- `misconception_timeline_figure` — 오개념 등장·소거 간트


In [ ]:
# 셀 8: 종합 시각화
from IPython.display import Markdown, display
from lib.visualize import (
    user_model_markdown, plot_radar_all, plot_user_history,
    cps_heatmap_figure, misconception_timeline_figure,
)
import matplotlib.pyplot as plt

# (a) Markdown 요약
display(Markdown(user_model_markdown(CONFIG, learner_models)))

# (b) 레이더
plot_radar_all(CONFIG, learner_models)

# (c) 변화 추이
plot_user_history(CONFIG, learner_models)

# (d) CPS 히트맵
fig_cps = cps_heatmap_figure(CONFIG, learner_models)
plt.show()

# (e) 오개념 타임라인
fig_mis = misconception_timeline_figure(CONFIG, learner_models)
plt.show()


---

## 🗂️ (선택) 레거시 CLI 런너

Gradio UI 대신 터미널 상호작용으로 세션을 돌리고 싶을 때.

명령어: `/radar` · `/history` · `/model` · `/next` · `/decision` · `/quit`


In [ ]:
# 셀 9 (선택): CLI 세션
from lib import run_session

# run_session(**ctx)   # 주석 해제 후 실행
